In [1]:
import json 
import pandas as pd 
import gurobipy as gp
from src.build_recipe_matrix import load_recipe_data
from src.recipe_data import recipes

In [2]:

recipe_df, dishes, ingredients = load_recipe_data()
dishes = [dish for dish, _ in recipes]

import json

with open("./data/hp_data.json", "r") as file:
    hp_data = json.load(file)

In [3]:
MAX_HP = 50

hp = {
    dish: data["percent"] * MAX_HP + data["flat"]
    for dish, data in hp_data.items()
}

print(hp)

{'Sweet Madame': 910.0, 'Mondstadt Hash Brown': 615.0, 'Matsutake Meat Rolls': 910.0, 'Chicken-Mushroom Skewer': 804.0, 'Grilled Tiger Fish': 804.0, 'Universal Peace': 615.0, 'Squirrel Fish': 615.0, 'Crystal Shrimp': 1809.0, 'Barbeque Ribs': 910.0, 'Dry-Braised Salted Fish': 804.0, 'Duck Confit': 804.0, 'Fish With Cream Sauce': 1809.0, 'Grilled Fish in Mint Sauce': 804.0, 'Grilled Unagi Fillet': 910.0, 'Lakkaberry Madame': 910.0, 'Mint Jelly': 804.0, 'Pita Pocket': 615.0, 'Radish Veggie Soup': 1264.0, 'Smoked Fish Steak': 804.0, 'Soba Noodles': 1264.0, 'Tasses Ragout': 910.0, 'Tea-Smoked Squab': 910.0, 'Tonkotsu Ramen': 615.0, 'Tricolor Dango': 615.0, 'Tuna Sushi': 910.0, 'Veggie Pot Soup': 615.0, 'Poisson Seafood Soup': 2713.0, 'Golden Tempered Jade': 1810.0, 'Northern Apple Stew': 615.0, 'Crispy Potato Shrimp Platter': 1363.0, 'Tahchin': 1363.0, 'Flaming Red Bolognese': 1809.0, 'Bulle Soufle': 3734.0, 'Fine Tea, Full Moon': 615.0}


In [4]:

with open("./data/inventory.json", "r") as file:
    Inventory = json.load(file)

In [5]:
import gurobipy as gp
model = gp.Model("Food production plan")
model.setParam("OutputFlag", 0)


hp_value = {
    dish: data["percent"] * 100 + data["flat"]
    for dish, data in hp_data.items()
}

x = model.addVars(
    dishes,
    vtype=gp.GRB.INTEGER,
    #vtype=gp.GRB.CONTINUOUS,
    lb=0,
    name="x"
)

Set parameter Username
Set parameter LicenseID to value 2845441


In [6]:
model.setObjective(
    gp.quicksum(
        hp_value[dish] * x[dish]
        for dish in dishes
    ), 
    gp.GRB.MAXIMIZE
)



In [7]:
constraints = {}

for ingredient in recipe_df.index:

    constraints[ingredient] = model.addConstr(
        gp.quicksum(
            recipe_df.loc[ingredient, dish] * x[dish]
            for dish in dishes
        )
        <= Inventory[ingredient],

        name=f"{ingredient}_inventory"
    )

In [8]:
model.write("food_allocation.lp")

In [9]:
model.optimize()
if model.status == gp.GRB.OPTIMAL:

    print("Maximum HP:", model.ObjVal)

    for dish in dishes:
        if x[dish].X > 0:
            print(
                dish,
                round(x[dish].X)
            )
    baseline_hp = model.ObjVal

Maximum HP: 369166.0
Mondstadt Hash Brown 10
Matsutake Meat Rolls 2
Chicken-Mushroom Skewer 46
Golden Tempered Jade 54
Barbeque Ribs 30
Grilled Fish in Mint Sauce 26
Radish Veggie Soup 105
Soba Noodles 7
Tricolor Dango 10
Veggie Pot Soup 6
Bulle Soufle 5
Northern Apple Stew 1
Crispy Potato Shrimp Platter 4


In [10]:
for ingredient in recipe_df.index:
    used = sum(
        recipe_df.loc[ingredient, dish] * x[dish].X
        for dish in dishes
    )
    
    available = Inventory[ingredient]
    utilisation = used / available if available > 0 else 0
    
    print(
        ingredient,
        f"{used:.0f}/{available}",
        f"({utilisation:.1%})"
    )

Apple 3/5 (60.0%)
Bamboo Shoot 0/54 (0.0%)
Berry 8/812 (1.0%)
Bird Egg 10/72 (13.9%)
Bulle 15/25 (60.0%)
Butter 1/3 (33.3%)
Cabbage 18/239 (7.5%)
Carrot 18/33 (54.5%)
Chenyu Adeptea 0/52 (0.0%)
Fish 26/26 (100.0%)
Flour 7/7 (100.0%)
Fowl 100/100 (100.0%)
Jam 10/10 (100.0%)
Lakkaberry 0/72 (0.0%)
Lotus Head 0/73 (0.0%)
Marcotte 30/59 (50.8%)
Matsutake 4/5 (80.0%)
Milk 30/83 (36.1%)
Mint 147/147 (100.0%)
Mushroom 46/67 (68.7%)
Onion 0/28 (0.0%)
Padisarah 0/5 (0.0%)
Pepper 61/61 (100.0%)
Pinecone 20/251 (8.0%)
Potato 34/35 (97.1%)
Radish 105/105 (100.0%)
Raw Meat 67/174 (38.5%)
Rice 10/10 (100.0%)
Sakura Bloom 20/168 (11.9%)
Salt 54/54 (100.0%)
Seagrass 7/467 (1.5%)
Shrimp Meat 16/161 (9.9%)
Smetana 12/18 (66.7%)
Snapdragon 20/109 (18.3%)
Sugar 5/5 (100.0%)
Sweet Flower 0/124 (0.0%)
Tomato 0/154 (0.0%)
Unagi Meat 0/10 (0.0%)


# Sensitivity analysis

In [11]:
sensitivity_model = gp.Model("Food Resource Allocation - Sensitivity")

sensitivity_x = sensitivity_model.addVars(
    dishes,
    vtype=gp.GRB.CONTINUOUS,
    lb=0,
    name="x"
)

sensitivity_model.setObjective(
    gp.quicksum(
        hp_value[dish] * sensitivity_x[dish]
        for dish in dishes
    ),
    gp.GRB.MAXIMIZE
)

sensitivity_constraints = {}

for ingredient in recipe_df.index:
    sensitivity_constraints[ingredient] = sensitivity_model.addConstr(
        gp.quicksum(
            recipe_df.loc[ingredient, dish] * sensitivity_x[dish]
            for dish in dishes
        )
        <= Inventory[ingredient],
        name=f"{ingredient}_inventory"
    )

sensitivity_model.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 23.2.0 23C64)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 38 rows, 34 columns and 99 nonzeros (Max)
Model fingerprint: 0xadbd81f0
Model has 34 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 4e+00]
  Objective range  [6e+02, 4e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [3e+00, 8e+02]

Presolve removed 32 rows and 19 columns
Presolve time: 0.00s
Presolved: 6 rows, 15 columns, 24 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.3134400e+05   2.020821e+01   0.000000e+00      0s
       3    3.7005433e+05   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.00 seconds (0.00 work units)
Optimal objective  3.700543333e+05


In [12]:
if sensitivity_model.status == gp.GRB.OPTIMAL:

    print("Continuous LP Maximum HP:", sensitivity_model.ObjVal)

    print("\n----- Shadow Prices -----")

    for ingredient, constraint in sensitivity_constraints.items():
        print(
            ingredient,
            round(constraint.Pi, 2)
        )

Continuous LP Maximum HP: 370054.3333333334

----- Shadow Prices -----
Apple 56.67
Bamboo Shoot 0.0
Berry 0.0
Bird Egg 0.0
Bulle 0.0
Butter 0.0
Cabbage 0.0
Carrot 0.0
Chenyu Adeptea 0.0
Fish 700.25
Flour 1268.0
Fowl 808.0
Jam 315.0
Lakkaberry 0.0
Lotus Head 0.0
Marcotte 0.0
Matsutake 460.0
Milk 0.0
Mint 107.75
Mushroom 0.0
Onion 0.0
Padisarah 0.0
Pepper 460.0
Pinecone 0.0
Potato 315.0
Radish 1160.25
Raw Meat 0.0
Rice 630.0
Sakura Bloom 0.0
Salt 1012.0
Seagrass 0.0
Shrimp Meat 0.0
Smetana 0.0
Snapdragon 0.0
Sugar 3748.0
Sweet Flower 0.0
Tomato 0.0
Unagi Meat 0.0
